# ML-10 · Content action playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

> **Freestyle submission.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**What this card asks:** ranked actions with reason codes, an archetype-to-action map, the key decay insight, intended
use, limits, human review, cost and value, monitoring and retrain triggers, and what must not be automated. **The
principle it serves:** rules give the reason code, models give the order, and a person decides.

## 1. Intended use

- **Who:** a FlyRank content editor, once a month, with a budget of 50 reviews.
- **When:** day 1 of the month, using the two closed months before it.
- **What it is:** a decision-support queue of top-10 pages most likely to lose at least 25% of their expected clicks at
  their current rank in the next two months, each with one reason code and one suggested action.
- **What it is not:** an automatic rewrite, a forecast of Google, or a causal claim that the action restores clicks.

In [1]:
# Setup: works locally and in Colab. Installs only missing packages and never prints a token.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "huggingface_hub", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
DATA = wh.data_available(try_download=True)   # needs HF_TOKEN as an env var or Colab secret
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8])
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

pinned warehouse snapshot available: True | revision 50cbf7c3


In [2]:
if DATA:
    import erosion_playbook; erosion_playbook.main()
pb = receipt("erosion_playbook_summary.json")
print({k: pb[k] for k in ["decision_origin", "feature_window", "trained_on_origins", "eligible_pages", "excluded_younger_than_90_days",
                          "queue_size", "queue_clients", "client_cap", "reason_codes", "clicks_at_risk_2_months_if_25pct_lost"]})

{
 "decision_origin": "2026-07",
 "feature_window": [
  "2026-05",
  "2026-06"
 ],
 "trained_on_origins": [
  "2025-12",
  "2026-01",
  "2026-02",
  "2026-03",
  "2026-04",
  "2026-05"
 ],
 "train_rows": 49022,
 "lr_C": 1.0,
 "eligible_pages": 14100,
 "excluded_younger_than_90_days": 3243,
 "queue_size": 50,
 "queue_clients": 7,
 "max_pages_one_client": 10,
 "reason_codes": {
  "CTR_ABOVE_PEERS": 40,
  "DEMAND_SHIFT": 10
 },
 "queue_median_position": 4.8,
 "clicks_at_risk_2_months_if_25pct_lost": 25324,
 "score_quantiles_eligible": {
  "0.5": 0.2291,
  "0.9": 0.4285,
  "0.99": 0.8354
 },
 "queue_min_score": 0.8037,
 "client_cap": 10,
 "cap_cost_forward_origins": {
  "2026-02": {
   "uncapped_p@50": 0.52,
   "capped_p@50": 0.54,
   "uncapped_clients": 6,
   "capped_clients": 8
  },
  "2026-03": {
   "uncapped_p@50": 0.6,
   "capped_p@50": 0.62,
   "uncapped_clients": 11,
   "capped_clients": 11
  },
  "2026-04": {
   "uncapped_p@50": 0.66,
   "capped_p@50": 0.66,
   "uncapped_clients": 

## 2. The ranked queue for the July 2026 decision

The frozen logistic model, retrained on every month whose label had matured (December to May), scores 14,100 eligible
pages. Pages younger than 90 days are excluded and no client gets more than 10 slots. Rows are shown by rank and an
anonymous client index. `rank` is the model's rank before the guardrails, so gaps (for example, no ranks 1 to 3) are
pages removed for being younger than 90 days or for exceeding a client's cap. The full queue with IDs for the editor
is written to a gitignored CSV.

In [3]:
q = pd.read_csv(os.path.join(OUT, "erosion_queue_2026-07.csv")) if os.path.exists(os.path.join(OUT, "erosion_queue_2026-07.csv")) else None
if q is not None:
    import erosion_playbook as P
    view = P.capped_top(q[~q.excluded_young]).copy()
    view["client"] = pd.factorize(view.client_hash_id)[0]
    display(view[["rank", "client", "position", "score", "top_driver", "reason_code"]].head(15).reset_index(drop=True))

,rank,client,position,score,top_driver,reason_code
0,4,0,2.8,0.9879,ctr_vs_ref,CTR_ABOVE_PEERS
1,5,0,2.8,0.9879,ctr_vs_ref,CTR_ABOVE_PEERS
2,7,0,2.8,0.9859,ctr_vs_ref,CTR_ABOVE_PEERS
3,15,1,4.1,0.9526,ctr_vs_ref,CTR_ABOVE_PEERS
4,17,1,4.1,0.9507,ctr_vs_ref,CTR_ABOVE_PEERS
5,18,1,4.1,0.9500,ctr_vs_ref,CTR_ABOVE_PEERS
6,19,1,4.1,0.9482,ctr_vs_ref,CTR_ABOVE_PEERS
7,20,1,4.1,0.9482,ctr_vs_ref,CTR_ABOVE_PEERS
8,21,1,4.1,0.9473,ctr_vs_ref,CTR_ABOVE_PEERS
9,22,1,4.1,0.9467,ctr_vs_ref,CTR_ABOVE_PEERS


## 3. Reason code to action

| Reason code (largest model contribution) | What it means | Action |
|---|---|---|
| `CTR_ABOVE_PEERS` | the page earns more clicks than its position normally does | **Protect:** check the live results page for an AI answer; make the page the citable source (answer-first summary, FAQ, structured data) before editing anything else |
| `DEMAND_SHIFT` | impressions swung sharply in the last month | **Check demand first:** seasonality or tracking may explain the change; no rewrite until confirmed |
| `CTR_ALREADY_SLIPPING` | a CTR fall has started | rewrite title and description toward the click an AI answer cannot satisfy (comparison, pricing, tool, example) |
| `TOP_POSITION_EXPOSED` | rank alone drives the score | make the answer citable; monitor |
| `RANK_MOVING` | position is unstable | **hold:** re-score next month |
| `CLIENT_WIDE_DECLINE` | the whole site is losing CTR at a stable rank | escalate to the account lead; page edits alone will not fix it |

## 4. The insight behind the playbook

The pages that lose clicks next are not the ones already slipping. They are the ones **currently earning more than
their rank normally earns**. That excess click share is what an AI answer can take. So the first action is protective:
make the page the source an AI answer cites. It is not a rewrite after the loss.

## 5. Human review (required before any action)

1. Open the live results page for the page's main query and record whether an AI answer appears and whether the page is
   cited.
2. If impressions jumped threefold or more, or the page had data in only one month, confirm the demand is real before
   acting (the main false-alarm pattern in ML-09).
3. Respect a 30-day cooldown after any edit, and skip pages younger than 90 days.
4. The editor can override any row. Overrides are logged with a reason, and they become training signal.

## 6. Cost and value

In [4]:
sealed = receipt("erosion_model_metrics_sealed.json")["origins"]["2026-05"]
p_model, base = sealed["metrics"]["logreg"]["p@50"], sealed["base_rate"]
print(f"Sealed month: {p_model*50:.0f} of 50 queued pages eroded with the model vs about {base*50:.0f} for a random 50 pages.")
print(f"Editor time: 50 reviews x about 20 minutes = about 17 hours per month.")
print(f"Clicks at stake in the July queue if each page lost 25% over two months: about {pb['clicks_at_risk_2_months_if_25pct_lost']:,}.")
display(pd.DataFrame(pb["cap_cost_forward_origins"]).T)

Sealed month: 21 of 50 queued pages eroded with the model vs about 12 for a random 50 pages.
Editor time: 50 reviews x about 20 minutes = about 17 hours per month.
Clicks at stake in the July queue if each page lost 25% over two months: about 25,324.


,uncapped_p@50,capped_p@50,uncapped_clients,capped_clients
2026-02,0.52,0.54,6.0,8.0
2026-03,0.60,0.62,11.0,11.0
2026-04,0.66,0.66,9.0,11.0
2026-05,0.42,0.54,15.0,16.0


**Reading it.**
- **Value of the model:** on the sealed month, the queue holds about 9 more eroding pages per 50 reviews than random
  selection. That is the time saved.
- **Cost of the client cap:** nothing in the development months (+0.02, +0.02, 0.00). On the sealed month it would have
  raised P@50 from 0.42 to 0.54. I added the cap after unsealing, for fairness of the review budget, so that number is a
  post-hoc observation, not the sealed claim.

## 7. Monitoring and retrain triggers

| Trigger | Threshold | Response |
|---|---|---|
| A new month's label matures | monthly | retrain on all matured months; re-run the forward check |
| Precision of the last matured queue | below 1.3x that month's base rate | pause the queue and investigate drift |
| Base rate moves | outside 0.15 to 0.45 | check for a Google change or a tracking break before trusting scores |
| Client concentration | one client above 10 slots before the cap | review that client's data feed |
| AI-answer presence | editors record AI answers on reviewed pages | after 3 months, test whether recorded presence adds signal |

## 8. What must not be automated

- **Rewriting or publishing content from the queue.** A person reviews the live results page first.
- **Treating a false alarm as harmless.** Every edit risks the rank the page holds.
- **Claiming that an action restored clicks.** That needs a hold-out (for example, leave 10 of the 50 queued pages
  untouched each month and compare), and this study does not have one yet.
- **Client-level conclusions from page queues.** Client-wide declines go to the account lead.

## 9. Next iteration (pre-registered)

From the ML-09 error analysis, I fix these before the next live month and test them on it:
1. **Require activity in both feature months.**
2. **Log editors' AI-answer observations as a feature.**
3. **Run the 10-page hold-out to measure the effect of the actions.**

## 10. Exports

`work/outputs/erosion_queue_2026-07.csv` (gitignored, row level), `erosion_playbook_summary.json`,
`baseline_action_score.csv` (gitignored), `baseline_metrics.json` and the figures in `work/figures/` feed the paper.